# NetCDF to DGGS (H3) Conversion for Canada Climate Indices

This notebook converts NetCDF climate indices to DGGS H3 grids, determines optimal resolution, aggregates mean values, stores results in Zarr format, and generates a pydggsapi-compatible config.

## Workflow Overview

1. **Configuration**: Set up directories and configurable parameters
2. **Find NetCDF Files**: Locate all climate index NetCDF files
3. **Analyze Spatial Resolution**: Determine optimal H3 resolution from NetCDF grid spacing
4. **Process Climate Data**: Map NetCDF grids to H3 cells and aggregate to multiple resolutions
5. **Extract Metadata**: Get variable metadata from NetCDF attributes and web sources
6. **Create Zarr Store**: Save multi-resolution DGGS data with metadata attributes
7. **Generate PyDGGSAPI Config**: Build final configuration JSON for API integration

## Cache Management

To reprocess data, clear the following caches in the `outputs/cache/` directory:
- `climate_metadata.json` - Variable metadata cache
- `grid2h3_*.npz` - Grid to H3 cell mappings
- `*_h3_*.parquet` - Processed climate data per variable
- `climate_dggs.zarr/` - Final Zarr store
- `dggs-collection.json` - PyDGGSAPI configuration

**Note on H3 Index Format**: This notebook uses int64 representation for H3 indices to save space.
If you have existing parquet files with string H3 IDs, run `python convert_parquet_to_int64.py` to convert them in-place.

---


In [ ]:
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import datetime
import glob
import h3
import json
import numpy as np
import os
import pandas as pd
import requests
import tempfile
import time
import xarray as xr
import zarr


## Step 1: Configuration

Set up directories and configurable parameters.


In [58]:
# Configurable indices and base directory
CLIMATE_INDICES =  ["prcptot"]  # ["tx_max", "ice_days", "frost_days", "prcptot"]
SUBDIRECTORY_FILTER = "allyears"  # Only process files from this subdirectory (None for all)
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
BASE_DATA_DIR = os.path.join(NOTEBOOK_DIR, "data")
OUTPUTS_DIR = os.path.join(NOTEBOOK_DIR, "outputs")

def ensure_directory(path):
    """
    Create directory, following symlinks if they exist.
    """
    if os.path.islink(path):
        target = os.readlink(path)
        if not os.path.isabs(target):
            target = os.path.join(os.path.dirname(path), target)
        os.makedirs(target, exist_ok=True)
    else:
        os.makedirs(path, exist_ok=True)

ensure_directory(OUTPUTS_DIR)
CACHE_DIR = os.environ.get("CLIMATE_CACHE_DIR", os.path.join(OUTPUTS_DIR, "cache"))
ensure_directory(CACHE_DIR)

print(f"Data directory: {BASE_DATA_DIR}")
print(f"Outputs directory: {OUTPUTS_DIR}")
print(f"Cache directory: {CACHE_DIR}")


Data directory: /home/chamigfr/dev/ogc/ogc-dggs/ogc11-dggs/canada-climate/data
Outputs directory: /home/chamigfr/dev/ogc/ogc-dggs/ogc11-dggs/canada-climate/outputs
Cache directory: /tmp/climate


## Step 2: Find NetCDF Files


In [59]:
def find_netcdf_files(base_dir, indices, subdir_filter=None):
    """
    Find NetCDF files matching climate indices.

    Args:
        base_dir: Base directory to search
        indices: List of climate index names to match in filenames
        subdir_filter: Only include files from paths containing this subdirectory name (None = no filter)
    """
    files = []
    for root, _, fnames in os.walk(base_dir):
        # Apply subdirectory filter if specified
        if subdir_filter and subdir_filter not in root:
            continue
        for fname in fnames:
            if fname.endswith(".nc") and any(idx in fname for idx in indices):
                files.append(os.path.join(root, fname))
    return files

nc_files = find_netcdf_files(BASE_DATA_DIR, CLIMATE_INDICES, SUBDIRECTORY_FILTER)
print(f"Found {len(nc_files)} NetCDF files" + (f" in '{SUBDIRECTORY_FILTER}' subdirectory" if SUBDIRECTORY_FILTER else "") + ".")


Found 17 NetCDF files in 'allyears' subdirectory.


In [60]:
sorted([os.path.basename(p) for p in nc_files])


['prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc',
 'prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_02February.nc',
 'prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_03March.nc',
 'prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_04April.nc',
 'prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_05May.nc',
 'prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_06June.nc',
 'prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_07July.nc',
 'prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_08August.nc',
 'prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_09September.nc',
 'prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_10October.nc',
 'prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_perce

## Step 3: Analyze Spatial Resolution and Determine H3 Resolution


In [61]:
if not nc_files:
    raise RuntimeError("No NetCDF files found in the directory. Please check BASE_DATA_DIR and file availability.")

resolutions = []
for f in tqdm(nc_files, desc="Analyzing spatial resolution", unit="file"):
    ds = xr.open_dataset(f, decode_timedelta=False)
    lat = ds["lat"] if "lat" in ds else ds["latitude"]
    lon = ds["lon"] if "lon" in ds else ds["longitude"]
    # Estimate resolution (in degrees)
    lat_res = float(np.abs(lat[1] - lat[0]))
    lon_res = float(np.abs(lon[1] - lon[0]))
    # Approximate km using haversine formula
    mean_lat = float(lat.mean())
    earth_radius_km = 6371.0
    lat_km = lat_res * (np.pi/180) * earth_radius_km
    lon_km = lon_res * (np.pi/180) * earth_radius_km * np.cos(mean_lat * np.pi/180)
    resolutions.append((f, lat_km, lon_km))
    ds.close()

Analyzing spatial resolution:   0%|          | 0/17 [00:00<?, ?file/s]

In [62]:
# Find best (finest) resolution
best_file, best_lat_km, best_lon_km = min(resolutions, key=lambda x: min(x[1], x[2]))
print(f"Best spatial resolution: {best_lat_km:.2f} km x {best_lon_km:.2f} km from {best_file}")

# Map NetCDF Grid to H3 DGGS
H3_RESOLUTION = None
for res in range(16):
    h3_edge_km = h3.average_hexagon_edge_length(res, "km")
    if h3_edge_km <= min(best_lat_km, best_lon_km):
        H3_RESOLUTION = res
        break
if H3_RESOLUTION is None:
    H3_RESOLUTION = 15
    h3_edge_km = h3.average_hexagon_edge_length(H3_RESOLUTION, "km")
else:
    h3_edge_km = h3.average_hexagon_edge_length(H3_RESOLUTION, "km")
print(f"Selected H3 resolution: {H3_RESOLUTION} (edge ~{h3_edge_km:.3f} km)")


Best spatial resolution: 9.27 km x 4.31 km from /home/chamigfr/dev/ogc/ogc-dggs/ogc11-dggs/canada-climate/data/allyears/prcptot/MS/prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc
Selected H3 resolution: 6 (edge ~3.725 km)


## Step 4: Process Climate Data to H3 DGGS

Map NetCDF grids to H3 cells, aggregate by cell and time, and save intermediate results.

### Grid to H3 Mapping with Caching


In [63]:
def cache_grid_to_h3(lat, lon, resolution, cache_dir=CACHE_DIR):
    """
    Map grid coordinates to H3 cells with caching.
    Cache file is based on lat/lon hash and resolution, so identical grids reuse the same mapping.
    Returns H3 indices as int64 for efficient storage.
    """
    # Convert DataArray to numpy array if needed
    lat_vals = lat.values if hasattr(lat, 'values') else np.array(lat)
    lon_vals = lon.values if hasattr(lon, 'values') else np.array(lon)

    # Create cache filename based on grid hash
    cache_file = os.path.join(cache_dir, f"grid2h3_{resolution}_{hash(tuple(lat_vals))}_{hash(tuple(lon_vals))}.npz")
    if os.path.exists(cache_file):
        data = np.load(cache_file, allow_pickle=True)
        return data["h3_indices"]

    h3_indices = np.empty((len(lat_vals), len(lon_vals)), dtype=np.int64)
    for i, la in enumerate(lat_vals):
        for j, lo in enumerate(lon_vals):
            h3_cell = h3.latlng_to_cell(float(la), float(lo), resolution)
            h3_indices[i, j] = int(h3_cell, 16)  # Convert hex string to int64
    np.savez_compressed(cache_file, h3_indices=h3_indices)
    return h3_indices


### Process NetCDF Files


In [ ]:
print(f"Processing {len(nc_files)} NetCDF files...")

# Initialize metadata collection
variable_metadata = {idx: {
    "description": "",
    "unit": "",
    "long_name": "",
    "standard_name": "",
    "url": ""
} for idx in CLIMATE_INDICES}

for f in tqdm(nc_files, desc="NetCDF files", unit="file"):
    ds = xr.open_dataset(f, decode_timedelta=False)

    # Find all variables matching our climate indices
    # Variables are named like: ssp126_prcptot_p50, ssp245_tx_max_delta_1991_2020_p10, etc.
    matching_vars = []
    for var_name in ds.data_vars:
        # Check if any of our climate indices is in the variable name
        for idx in CLIMATE_INDICES:
            if idx in var_name:
                matching_vars.append(var_name)

                # Extract metadata for this climate index (if not already found)
                if not variable_metadata[idx]["long_name"]:
                    var = ds[var_name]
                    attrs = var.attrs

                    if "long_name" in attrs:
                        variable_metadata[idx]["long_name"] = attrs["long_name"]
                    if "description" in attrs:
                        variable_metadata[idx]["description"] = attrs["description"]
                    if not variable_metadata[idx]["description"] and "long_name" in attrs:
                        variable_metadata[idx]["description"] = attrs["long_name"]
                    if "units" in attrs:
                        variable_metadata[idx]["unit"] = attrs["units"]
                    elif "unit" in attrs:
                        variable_metadata[idx]["unit"] = attrs["unit"]
                    if "standard_name" in attrs:
                        variable_metadata[idx]["standard_name"] = attrs["standard_name"]

                break

    if not matching_vars:
        tqdm.write(f"No matching variables in {os.path.basename(f)}")
        ds.close()
        continue

    # Get grid coordinates once per file
    lat = ds["lat"] if "lat" in ds else ds["latitude"]
    lon = ds["lon"] if "lon" in ds else ds["longitude"]
    h3_indices = cache_grid_to_h3(lat, lon, H3_RESOLUTION)
    time_dim = ds["time"] if "time" in ds else ds["year"]

    # Process each matching variable
    for var_name in tqdm(matching_vars, desc=f"Variables in {os.path.basename(f)}", leave=False, unit="var"):
        temp_file = os.path.join(OUTPUTS_DIR, f"{var_name}_h3_{H3_RESOLUTION}.parquet")
        if os.path.exists(temp_file):
            tqdm.write(f"Using cached {var_name} data")
            continue

        tqdm.write(f"Processing {var_name}...")
        values = ds[var_name].values
        records = []

        # Convert time dimension to pandas datetime once
        time_values = pd.to_datetime(time_dim.values)

        for t_idx in tqdm(range(len(time_dim)), desc=f"Time steps ({var_name})", leave=False, unit="step"):
            grid_vals = values[t_idx] if values.ndim == 3 else values
            for i in range(len(lat)):
                for j in range(len(lon)):
                    h3_id = h3_indices[i, j]
                    val = float(grid_vals[i, j])
                    # Skip NaN values
                    if not np.isnan(val):
                        records.append({
                            "h3_id": h3_id,
                            "datetime": time_values[t_idx],
                            var_name: val
                        })

        if records:
            df = pd.DataFrame(records)
            df = df.groupby(["h3_id", "datetime"])[var_name].mean().reset_index()
            df.to_parquet(temp_file)
            tqdm.write(f"Saved {var_name} to parquet ({len(df)} records)")
        else:
            tqdm.write(f"Warning: No valid data for {var_name}")

    ds.close()
print("Data processing complete")


Processing 17 NetCDF files...


NetCDF files:   0%|          | 0/17 [00:00<?, ?file/s]

Variables in prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc:   0%| …

Using cached ssp126_prcptot_p10 data
Processing ssp126_prcptot_p50...


Time steps (ssp126_prcptot_p50):   0%|          | 0/151 [00:00<?, ?step/s]

## Step 5: Supplement Metadata with Web URLs

Metadata has been extracted from NetCDF files during processing. Now supplement with URLs from climatedata.ca.


In [ ]:
def supplement_metadata_urls(metadata, cache_dir=CACHE_DIR):
    """
    Supplement metadata with URLs from climatedata.ca/variables/.
    """
    cache_file = os.path.join(cache_dir, "climate_metadata.json")

    # Check if we have cached metadata with URLs
    if os.path.exists(cache_file):
        print(f"Loading metadata from cache: {cache_file}")
        with open(cache_file, mode="r", encoding="utf-8") as f:
            cached = json.load(f)
            # Use cached URLs if available
            for idx in metadata:
                if idx in cached and cached[idx].get("url"):
                    metadata[idx]["url"] = cached[idx]["url"]
            return metadata

    # Fetch slug mapping from web
    try:
        url = "https://climatedata.ca/variables/"
        print(f"Fetching variable slugs from {url}...")
        resp = requests.get(url, timeout=15, allow_redirects=True)

        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "html.parser")
            var_links = soup.find_all("a", href=lambda x: x and "/variable/" in x)

            slug_map = {}
            for link in var_links:
                href = link.get("href", "")
                if "/variable/" in href:
                    slug = href.split("/variable/")[-1].rstrip("/")
                    if slug:
                        full_url = href if href.startswith("http") else f"https://climatedata.ca{href}"
                        slug_map[slug] = full_url

            print(f"Found {len(slug_map)} variable slugs")

            # Match URLs to our climate indices
            for idx in metadata:
                if idx in slug_map:
                    metadata[idx]["url"] = slug_map[idx]
                else:
                    hyphenated = idx.replace("_", "-")
                    if hyphenated in slug_map:
                        metadata[idx]["url"] = slug_map[hyphenated]
        else:
            print(f"Warning: Could not fetch variables list (status {resp.status_code})")
    except Exception as e:
        print(f"Warning: Could not fetch variables listing: {e}")

    # Set fallback URLs for any missing
    for idx in metadata:
        if not metadata[idx]["url"]:
            metadata[idx]["url"] = f"https://climatedata.ca/variable/{idx.replace('_', '-')}/"

    # Cache the complete metadata
    with open(cache_file, mode="w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    return metadata


In [ ]:
start_time = time.time()
variable_metadata = supplement_metadata_urls(variable_metadata)
elapsed = time.time() - start_time
print(f"Metadata URL supplementation completed in {elapsed:.2f} seconds")
print("Final metadata:", json.dumps(variable_metadata, indent=2))


## Step 6: Create Zarr Store with Metadata

Aggregate data to lower resolutions and save in Zarr format with metadata attributes.

### Aggregation Function


In [ ]:
def aggregate_to_lower_res(df, start_res, min_res):
    """
    Aggregate DGGS data from higher resolution to lower resolutions.
    Each step up reduces resolution by aggregating child cells to parent cells.
    H3 indices are stored as int64.
    """
    dfs = {start_res: df}
    for res in range(start_res-1, min_res-1, -1):
        df_res = df.copy()
        # Convert int64 to H3 address, get parent, convert back to int64
        df_res["h3_id"] = df_res["h3_id"].apply(
            lambda h: int(h3.cell_to_parent(h3.int_to_str(h), res), 16)
        )
        df_res = df_res.groupby(["h3_id", "datetime"]).mean().reset_index()
        dfs[res] = df_res
    return dfs


### Create Zarr Arrays


In [ ]:
zarr_store_path = os.path.join(OUTPUTS_DIR, "climate_dggs.zarr")
root = zarr.open_group(zarr_store_path, mode="w")

# Find all processed parquet files
parquet_files = glob.glob(os.path.join(OUTPUTS_DIR, f"*_h3_{H3_RESOLUTION}.parquet"))
processed_vars = [os.path.basename(f).replace(f"_h3_{H3_RESOLUTION}.parquet", "") for f in parquet_files]

print(f"Found {len(processed_vars)} processed variables")

for var in tqdm(processed_vars, desc="Creating Zarr arrays", unit="var"):
    temp_file = os.path.join(OUTPUTS_DIR, f"{var}_h3_{H3_RESOLUTION}.parquet")
    if not os.path.exists(temp_file):
        tqdm.write(f"Skipping {var} - no processed data found")
        continue

    tqdm.write(f"Creating Zarr arrays for {var}...")
    df = pd.read_parquet(temp_file)
    dfs = aggregate_to_lower_res(df, H3_RESOLUTION, 0)

    for res, df_res in dfs.items():
        grp = root.require_group(f"res{res}")
        arr = grp.create_array(var, shape=df_res[var].shape, dtype="float32", chunks=(10000,), overwrite=True)
        arr[:] = df_res[var].values
        h3_id_arr = grp.create_array(f"h3_id_{var}", shape=df_res["h3_id"].shape, dtype="int64", overwrite=True)
        h3_id_arr[:] = df_res["h3_id"].values
        dt_arr = grp.create_array(f"datetime_{var}", shape=df_res["datetime"].shape, dtype="S25", overwrite=True)
        dt_arr[:] = df_res["datetime"].values.astype("str")

        # Match variable to base climate index for metadata
        base_idx = None
        for idx in CLIMATE_INDICES:
            if idx in var:
                base_idx = idx
                break

        if base_idx and base_idx in variable_metadata:
            arr.attrs["description"] = variable_metadata[base_idx]["description"]
            arr.attrs["unit"] = variable_metadata[base_idx]["unit"]
            arr.attrs["source_url"] = variable_metadata[base_idx]["url"]

        # Add variable-specific attributes
        arr.attrs["full_variable_name"] = var

print(f"Zarr store created at {zarr_store_path}")


## Step 7: Generate PyDGGSAPI Configuration

Compute spatial and temporal extents, then build the final configuration JSON.

### Compute Extents


In [ ]:
def compute_spatial_bbox(nc_files):
    """
    Compute spatial bounding box from all NetCDF files.
    """
    min_lat, max_lat, min_lon, max_lon = None, None, None, None
    for f in tqdm(nc_files, desc="Computing spatial bbox", leave=False, unit="file"):
        ds = xr.open_dataset(f, decode_timedelta=False)
        lat = ds["lat"] if "lat" in ds else ds["latitude"]
        lon = ds["lon"] if "lon" in ds else ds["longitude"]
        lat_min, lat_max = float(lat.min()), float(lat.max())
        lon_min, lon_max = float(lon.min()), float(lon.max())
        if min_lat is None or lat_min < min_lat:
            min_lat = lat_min
        if max_lat is None or lat_max > max_lat:
            max_lat = lat_max
        if min_lon is None or lon_min < min_lon:
            min_lon = lon_min
        if max_lon is None or lon_max > max_lon:
            max_lon = lon_max
        ds.close()
    return [min_lon, min_lat, max_lon, max_lat]

def compute_temporal_range(nc_files):
    """
    Compute temporal range from all NetCDF files.
    """
    min_time, max_time = None, None
    for f in tqdm(nc_files, desc="Computing temporal range", leave=False, unit="file"):
        ds = xr.open_dataset(f, decode_timedelta=False)
        if "time" in ds:
            times = ds["time"].values
        elif "year" in ds:
            times = ds["year"].values
        else:
            ds.close()
            continue
        t_min, t_max = pd.to_datetime(times).min(), pd.to_datetime(times).max()
        if min_time is None or t_min < min_time:
            min_time = t_min
        if max_time is None or t_max > max_time:
            max_time = t_max
        ds.close()
    return [str(min_time), str(max_time)]


In [ ]:
spatial_bbox = compute_spatial_bbox(nc_files)
temporal_range = compute_temporal_range(nc_files)
print(f"Spatial bbox: {spatial_bbox}")
print(f"Temporal range: {temporal_range}")


### Build Configuration JSON


In [ ]:
def build_dggs_collection_json(temporal_range, variables, zarr_store_path, spatial_bbox, h3_resolution):
    today = datetime.date.today().isoformat()
    collection_json = {
        "collections": {
            "canada-climatedata": {
                "title": "Canada Climate Indices DGGS Collection",
                "description": "Climate indices mapped to H3 DGGS cells, aggregated from NetCDF sources provided by ClimateData.ca.",
                "attribution": f"ClimateData.ca [Accessed on {today}]",
                "extent": {
                    "spatial": {
                        "bbox": [spatial_bbox],
                        "crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84"
                    },
                    "temporal": {
                        "interval": [temporal_range],
                        "trs": "http://www.opengis.net/def/uom/ISO-8601/0/Gregorian"
                    }
                },
                "links": [
                    {
                        "rel": "about",
                        "href": "https://climatedata.ca/",
                        "type": "text/html",
                        "title": "ClimateData.ca - Canadian Climate Data Portal",
                        "hreflang": "en-CA"
                    },
                    {
                        "rel": "about",
                        "href": "https://donneesclimatiques.ca/",
                        "type": "text/html",
                        "title": "Portail canadien des données climatiques",
                        "hreflang": "fr-CA"
                    },
                    {
                        "rel": "license",
                        "href": "https://creativecommons.org/licenses/by/4.0/legalcode",
                        "type": "text/html",
                        "title": "Creative Commons Attribution International (CC BY)",
                        "hreflang": "en-CA"
                    },
                    {
                        "rel": "terms-of-service",
                        "href": "https://climatedata.ca/about/legal/terms/",
                        "type": "text/html",
                        "title": "ClimateData.ca Terms of Service",
                        "hreflang": "en-CA"
                    },
                    {
                        "rel": "describedby",
                        "href": "https://climatedata.ca/variables/",
                        "type": "text/html",
                        "title": "ClimateData.ca Variables Documentation",
                        "hreflang": "en-CA"
                    },
                    {
                        "rel": "convertedfrom",
                        "href": "https://climatedata.ca/",
                        "type": "text/html",
                        "title": "Original ClimateData.ca Dataset Portal",
                        "hreflang": "en-CA"
                    },
                    {
                        "rel": "via",
                        "href": "https://github.com/crim-ca/ogc-dggs/tree/main/canada-climate",
                        "type": "text/html",
                        "title": "Source code employed to generate DGGS data from reference features",
                        "hreflang": "en-CA"
                    },
                    {
                        "rel": "cite-as",
                        "href": "https://hirondelle.crim.ca/dggs-api/collections/canada-climatedata",
                        "type": "application/json",
                        "title": "Canada Climate Indices as DGGS Collection",
                        "hreflang": "en-CA"
                    },
                    {
                        "rel": "sponsored",
                        "href": "https://climatedata.ca/about/citing-climatedata-ca/",
                        "type": "text/html",
                        "title": "ClimateData.ca Sponsorship and Funding",
                        "hreflang": "en-CA"
                    },
                    {
                        "rel": "author",
                        "href": "https://crim.ca/fr/",
                        "type": "text/html",
                        "hreflang": "fr-CA",
                        "title": "Centre de recherche informatique de Montréal (CRIM)"
                    },
                    {
                        "rel": "author",
                        "href": "https://crim.ca/en/",
                        "type": "text/html",
                        "hreflang": "en-CA",
                        "title": "Centre de recherche informatique de Montréal (CRIM)"
                    }
                ],
                "collection_provider": {
                    "providerId": "canada-climatedata",
                    "dggrsId": "H3",
                    "dggrs_zoneid_repr": "int",
                    "min_refinement_level": 0,
                    "max_refinement_level": h3_resolution,
                    "datasource_id": "canada-climatedata"
                }
            }
        },
        "collection_providers": {
            "canada-climatedata": {
                "classname": "zarr_collection_provider.ZarrCollectionProvider",
                "datasources": {
                    "canada-climatedata": {
                        "filepath": zarr_store_path,
                        "id_col": "h3_id",
                        "datetime_col": "datetime",
                        "data_cols": variables
                    }
                }
            }
        },
        "dggrs": {
            "H3": {
                "title": "DGGRS H3",
                "description": "H3 hexagonal hierarchical geospatial indexing system developed by Uber.",
                "crs": "wgs84",
                "shapeType": "hexagon",
                "uri": "https://www.opengis.net/def/dggrs/OGC/1.0/H3",
                "definition_link": "https://www.opengis.net/def/dggrs/OGC/1.0/H3",
                "defaultDepth": 0,
                "classname": "h3_dggrs_provider.H3Provider"
            }
        }
    }
    return collection_json


In [ ]:
# Get list of all processed variables
parquet_files = glob.glob(os.path.join(OUTPUTS_DIR, f"*_h3_{H3_RESOLUTION}.parquet"))
processed_vars = [os.path.basename(f).replace(f"_h3_{H3_RESOLUTION}.parquet", "") for f in parquet_files]

collection_json = build_dggs_collection_json(
    temporal_range,
    processed_vars,  # Use actual processed variables, not just base indices
    zarr_store_path,
    spatial_bbox,
    H3_RESOLUTION  # Pass dynamic resolution
)

config_path = os.path.join(OUTPUTS_DIR, "dggs-collection.json")
with open(config_path, mode="w", encoding="utf-8") as f:
    json.dump(collection_json, f, indent=2)
print(f"Config saved to {config_path}")
print(
    f"Configured {len(processed_vars)} variables: "
    f"{', '.join(processed_vars[:5])}{'...' if len(processed_vars) > 5 else ''}"
)
